# FedTrace — 03: Three-Number Record Assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fedtrace/fedtrace.github.io/blob/main/notebooks/03_assembly.ipynb)

**Runtime:** CPU only  
**Estimated time:** ~5 min  
**Input checkpoints:** `data/raw/02_*.jsonl` — pulled from GitHub via `prepare_notebook` (written by notebook 02)  
**Publishes to GitHub:** `data/outputs/03_assembly.json`, `data/findings/03_assembly.md`

**Driving questions:**
1. For all matched contracts, what is the distribution of the ceiling-to-obligation gap? Does the IDV pattern (large headroom by design) hold at scale?
2. For what fraction of cancelled contracts was the obligated amount near zero at cancellation?
3. What is outlay coverage — for what fraction are all three numbers present?
4. How does the DOGE `savings` figure compare to the actual ceiling-to-obligation gap?
5. For matched grants, what is the financial record?
6. What are the aggregate totals across all cancelled awards — ceiling, obligated, outlays?

**Design:** assembly only — reads JSONL checkpoints from `data/raw/` (committed by notebook 02), joins IDV amounts, computes the three-number record for every matched award. No API calls.

**Methodology note:** DOGE `savings` = ceiling minus current obligations. Unexercised contractual headroom, not recovered funds. Termination costs for work-in-progress are not netted. Any single number in isolation is incomplete — ceiling, obligation, and outlay are not interchangeable.

Current findings: [`data/findings/03_assembly.md`](../data/findings/03_assembly.md).

## Setup

Run cells 1–4 at the start of every session. They are idempotent — safe to re-run.

In [ ]:
# ── CELL 1: Clone repo & install dependencies ─────────────────────────────────
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/fedtrace.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(
        ['git', 'clone', '--depth=1', 'https://github.com/fedtrace/fedtrace.github.io.git', str(REPO)],
        check=True, env=env,
    )
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('03_assembly', requirements_path=str(REPO / 'requirements.txt'))

In [ ]:
# ── CELL 2: Pull latest & set up paths ────────────────────────────────────────
from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
PATHS['outputs_dir'].mkdir(parents=True, exist_ok=True)
(PATHS['data_root'] / 'findings').mkdir(parents=True, exist_ok=True)
print(f'Repo ready at {REPO}')

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# ── CELL 4: Input and output paths ────────────────────────────────────────────
# Inputs — checkpoints written by notebook 02 (gitignored, local to Colab)
matched_path     = PATHS['raw_dir'] / '02_contracts_matched.jsonl'
unmatched_path   = PATHS['raw_dir'] / '02_contracts_unmatched.jsonl'
idv_amounts_path = PATHS['raw_dir'] / '02_idv_amounts.jsonl'
grants_path      = PATHS['raw_dir'] / '02_grants.jsonl'

# Outputs — pushed to GitHub by the publish cell
output_json_path   = PATHS['outputs_dir'] / '03_assembly.json'
findings_md_path   = PATHS['data_root'] / 'findings' / '03_assembly.md'

def _load_jsonl(path: Path, label: str) -> list[dict]:
    """Load a JSONL file; raise with a clear message if missing."""
    if not path.exists():
        raise FileNotFoundError(
            f'{label} checkpoint not found at {path}. '
            'Run notebook 02 to completion before running this notebook.'
        )
    records = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def _safe_pct(a, b) -> float:
    """Return a / b * 100, guarding against zero denominator."""
    return a / (b or 1) * 100

print('Checkpoint state:')
for label, path in [
    ('contracts matched',   matched_path),
    ('contracts unmatched', unmatched_path),
    ('IDV amounts',         idv_amounts_path),
    ('grants',              grants_path),
]:
    n = sum(1 for l in path.read_text().splitlines() if l.strip()) if path.exists() else 0
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  {label:25s}: {n:,} records  [{status}]')

## 1. Load Raw Data

In [ ]:
matched_records   = _load_jsonl(matched_path,     'contracts matched')
idv_amounts_recs  = _load_jsonl(idv_amounts_path, 'IDV amounts')
grants_records    = _load_jsonl(grants_path,      'grants')

# Unmatched is informational only — needed for total denominator
unmatched_records = _load_jsonl(unmatched_path, 'contracts unmatched') if unmatched_path.exists() else []

matched_df   = pd.DataFrame(matched_records)
idv_amts_df  = pd.DataFrame(idv_amounts_recs)
grants_df    = pd.DataFrame(grants_records)
unmatched_df = pd.DataFrame(unmatched_records) if unmatched_records else pd.DataFrame()

print('Record counts loaded from notebook 02 checkpoints:')
print(f'  matched_df:   {len(matched_df):,}')
print(f'  idv_amts_df:  {len(idv_amts_df):,}')
print(f'  grants_df:    {len(grants_df):,}')
print(f'  unmatched_df: {len(unmatched_df):,}')
if 'record_type' in matched_df.columns:
    rt_counts = matched_df['record_type'].value_counts()
    print(f'\nMatched by record_type:')
    for rt, n in rt_counts.items():
        print(f'  {rt}: {n:,}')

## 2. Contract Three-Number Record

**Deduplication rule:** the matched checkpoint is authoritative. Any PIID appearing in both matched and unmatched is treated as matched — unmatched rows are artifacts of crashed runs.

**Field sources:**
- `ceiling` — DOGE `value` field (contractual maximum including unexercised options)
- `obligated` — USASpending `Award Amount` (direct contracts) or `child_award_total_obligation` (IDVs)
- `outlays` — USASpending `Total Outlays` (direct contracts) or `child_award_total_outlay` (IDVs)

**IDV note:** For IDV records, the top-level USASpending obligation fields are $0 by design. All actual spending flows through child task orders. The IDV amounts endpoint returns aggregate child obligations and outlays.

In [ ]:
# Deduplication: matched set is authoritative
# A PIID in both matched and unmatched means a crash mid-batch — matched entry wins.
if not matched_df.empty and not unmatched_df.empty:
    matched_piids = set(matched_df.get('piid', matched_df.get('Award ID', pd.Series())).dropna())
    if 'piid' in unmatched_df.columns:
        overlap = matched_piids & set(unmatched_df['piid'].dropna())
        if overlap:
            print(f'Overlap (matched takes precedence): {len(overlap):,} PIIDs')

# Normalise the PIID column — matched records have Award ID from USASpending
if 'piid' not in matched_df.columns and 'Award ID' in matched_df.columns:
    matched_df = matched_df.rename(columns={'Award ID': 'piid'})
elif 'Award ID' in matched_df.columns and 'piid' in matched_df.columns:
    # Both present; prefer the DOGE piid column (populated in contract fetch merge)
    matched_df['piid'] = matched_df['piid'].fillna(matched_df['Award ID'])

# Merge IDV child amounts onto IDV rows
# Key: generated_internal_id (from USASpending search result, used as IDV amounts endpoint key)
if not idv_amts_df.empty and 'generated_internal_id' in matched_df.columns:
    idv_amts_df = idv_amts_df.rename(columns={
        'child_award_total_obligation': '_idv_obligated',
        'child_award_total_outlay':     '_idv_outlays',
    })
    # Keep only the columns needed for the join; drop duplicates on the key
    idv_join = idv_amts_df[['generated_internal_id', '_idv_obligated', '_idv_outlays']].drop_duplicates(
        subset='generated_internal_id', keep='last'
    )
    matched_df = matched_df.merge(idv_join, on='generated_internal_id', how='left')
    print(f'IDV amounts joined: {matched_df["_idv_obligated"].notna().sum():,} rows received child amounts')
else:
    matched_df['_idv_obligated'] = pd.NA
    matched_df['_idv_outlays']   = pd.NA
    print('No IDV amounts to join (either no IDVs or idv_amts_df is empty)')

# Resolve ceiling, obligated, outlays per record type
def _resolve_ceiling(row):
    """Ceiling = DOGE value field."""
    v = row.get('value')
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def _resolve_obligated(row):
    rt = row.get('record_type')
    if rt == 'idv':
        v = row.get('_idv_obligated')
    else:
        v = row.get('Award Amount')
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def _resolve_outlays(row):
    rt = row.get('record_type')
    if rt == 'idv':
        v = row.get('_idv_outlays')
    else:
        v = row.get('Total Outlays')
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

matched_df['ceiling']   = matched_df.apply(_resolve_ceiling,   axis=1)
matched_df['obligated'] = matched_df.apply(_resolve_obligated, axis=1)
matched_df['outlays']   = matched_df.apply(_resolve_outlays,   axis=1)

# Gap metrics
matched_df['ceiling_gap_pct'] = matched_df.apply(
    lambda r: (r['ceiling'] - r['obligated']) / (r['ceiling'] or 1) * 100
    if r['ceiling'] is not None and r['obligated'] is not None and r['ceiling'] > 0
    else None,
    axis=1,
)
matched_df['outlay_rate'] = matched_df.apply(
    lambda r: r['outlays'] / (r['obligated'] or 1)
    if r['outlays'] is not None and r['obligated'] is not None and r['obligated'] > 0
    else None,
    axis=1,
)

# Rename columns for output canonical schema
col_map = {
    'Recipient Name':                          'recipient_name',
    'Period of Performance Start Date':        'period_start',
    'Period of Performance Current End Date':  'period_end',
}
matched_df = matched_df.rename(columns={k: v for k, v in col_map.items() if k in matched_df.columns})

# Final output columns
out_cols = [
    'piid', 'agency', 'recipient_name',
    'ceiling', 'obligated', 'outlays',
    'record_type', 'ceiling_gap_pct', 'outlay_rate',
    'period_start', 'period_end',
]
contracts_out = matched_df[[c for c in out_cols if c in matched_df.columns]].copy()

print(f'\nContracts assembled: {len(contracts_out):,}')
print(f'  ceiling present:   {contracts_out["ceiling"].notna().sum():,}')
print(f'  obligated present: {contracts_out["obligated"].notna().sum():,}')
print(f'  outlays present:   {contracts_out["outlays"].notna().sum():,}')

## 3. Grant Three-Number Record

**Field sources:**
- `ceiling` — DOGE `value` field
- `obligated` — USASpending `total_obligation`
- `outlays` — USASpending `total_outlays`

No IDV join needed — grant awards have a flat structure.

In [ ]:
def _to_float(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

grants_df['ceiling']   = grants_df['value'].apply(_to_float) if 'value' in grants_df.columns else None
grants_df['obligated'] = grants_df['total_obligation'].apply(_to_float) if 'total_obligation' in grants_df.columns else None
grants_df['outlays']   = grants_df['total_outlays'].apply(_to_float) if 'total_outlays' in grants_df.columns else None

grants_df['ceiling_gap_pct'] = grants_df.apply(
    lambda r: (r['ceiling'] - r['obligated']) / (r['ceiling'] or 1) * 100
    if r.get('ceiling') is not None and r.get('obligated') is not None and r.get('ceiling', 0) > 0
    else None,
    axis=1,
)
grants_df['outlay_rate'] = grants_df.apply(
    lambda r: r['outlays'] / (r['obligated'] or 1)
    if r.get('outlays') is not None and r.get('obligated') is not None and r.get('obligated', 0) > 0
    else None,
    axis=1,
)

# Rename agency column if present
if 'awarding_agency' in grants_df.columns and 'agency' not in grants_df.columns:
    grants_df = grants_df.rename(columns={'awarding_agency': 'agency'})

grants_df['record_type'] = 'grant'

grant_out_cols = [
    'award_id', 'agency', 'recipient_name',
    'ceiling', 'obligated', 'outlays',
    'record_type', 'ceiling_gap_pct', 'outlay_rate',
    'period_start', 'period_end',
]
grants_out = grants_df[[c for c in grant_out_cols if c in grants_df.columns]].copy()

print(f'Grants assembled: {len(grants_out):,}')
print(f'  ceiling present:   {grants_out["ceiling"].notna().sum():,}')
print(f'  obligated present: {grants_out["obligated"].notna().sum():,}')
print(f'  outlays present:   {grants_out["outlays"].notna().sum():,}')

## 4. Coverage and Gap Analysis

In [ ]:
# ── Contract coverage ─────────────────────────────────────────────────────────
n_contracts = len(contracts_out)
n_grants    = len(grants_out)

# Three-number completeness
c_all_three = int((
    contracts_out['ceiling'].notna() &
    contracts_out['obligated'].notna() &
    contracts_out['outlays'].notna()
).sum()) if n_contracts else 0

g_all_three = int((
    grants_out['ceiling'].notna() &
    grants_out['obligated'].notna() &
    grants_out['outlays'].notna()
).sum()) if n_grants else 0

# Near-zero obligation at cancellation (obligated < 1% of ceiling)
c_near_zero_obl = 0
if n_contracts:
    mask = (
        contracts_out['ceiling'].notna() &
        contracts_out['obligated'].notna() &
        (contracts_out['ceiling'] > 0)
    )
    pct_obl = contracts_out.loc[mask, 'obligated'] / contracts_out.loc[mask, 'ceiling'].replace(0, float('nan'))
    c_near_zero_obl = int((pct_obl < 0.01).sum())

# Ceiling gap by record type
gap_summary = {}
if 'record_type' in contracts_out.columns and 'ceiling_gap_pct' in contracts_out.columns:
    for rt in contracts_out['record_type'].dropna().unique():
        subset = contracts_out.loc[
            (contracts_out['record_type'] == rt) & contracts_out['ceiling_gap_pct'].notna(),
            'ceiling_gap_pct'
        ]
        if not subset.empty:
            gap_summary[rt] = {
                'count': int(len(subset)),
                'median_gap_pct': round(float(subset.median()), 1),
                'mean_gap_pct':   round(float(subset.mean()),   1),
                'pct_gt_50':      round(_safe_pct((subset > 50).sum(), len(subset)), 1),
            }

# DOGE savings vs ceiling-to-obligation gap
doge_savings_col = None
for candidate in ['savings', 'doge_savings']:
    if candidate in matched_df.columns:
        doge_savings_col = candidate
        break

savings_comparison = {}
if doge_savings_col and 'ceiling' in contracts_out.columns and 'obligated' in contracts_out.columns:
    cmp = matched_df[[doge_savings_col, 'ceiling', 'obligated']].copy()
    cmp[doge_savings_col] = pd.to_numeric(cmp[doge_savings_col], errors='coerce')
    cmp['ceiling']   = pd.to_numeric(cmp['ceiling'],   errors='coerce')
    cmp['obligated'] = pd.to_numeric(cmp['obligated'], errors='coerce')
    cmp['actual_gap'] = cmp['ceiling'] - cmp['obligated']
    valid = cmp.dropna(subset=[doge_savings_col, 'actual_gap'])
    if not valid.empty:
        savings_comparison = {
            'n': int(len(valid)),
            'median_doge_savings':  round(float(valid[doge_savings_col].median()),  0),
            'median_actual_gap':    round(float(valid['actual_gap'].median()),       0),
            'total_doge_savings':   round(float(valid[doge_savings_col].sum()),      0),
            'total_actual_gap':     round(float(valid['actual_gap'].sum()),          0),
        }

# Aggregate totals — sum where present
def _sum_col(df, col):
    if col not in df.columns:
        return None
    s = pd.to_numeric(df[col], errors='coerce')
    return round(float(s.sum()), 0) if s.notna().any() else None

agg_contracts = {
    'total_ceiling':   _sum_col(contracts_out, 'ceiling'),
    'total_obligated': _sum_col(contracts_out, 'obligated'),
    'total_outlays':   _sum_col(contracts_out, 'outlays'),
}
agg_grants = {
    'total_ceiling':   _sum_col(grants_out, 'ceiling'),
    'total_obligated': _sum_col(grants_out, 'obligated'),
    'total_outlays':   _sum_col(grants_out, 'outlays'),
}
agg_all = {
    'total_ceiling':   (agg_contracts['total_ceiling'] or 0) + (agg_grants['total_ceiling'] or 0),
    'total_obligated': (agg_contracts['total_obligated'] or 0) + (agg_grants['total_obligated'] or 0),
    'total_outlays':   (agg_contracts['total_outlays'] or 0) + (agg_grants['total_outlays'] or 0),
}

# ── Print findings ─────────────────────────────────────────────────────────────
print('=== Coverage ===')
print(f'Contracts with all three numbers: {c_all_three:,} / {n_contracts:,} ({_safe_pct(c_all_three, n_contracts):.1f}%)')
print(f'Grants with all three numbers:    {g_all_three:,} / {n_grants:,} ({_safe_pct(g_all_three, n_grants):.1f}%)')
print(f'Contracts near-zero obligation (<1% of ceiling): {c_near_zero_obl:,}')
print()
print('=== Ceiling Gap by Record Type ===')
for rt, stats in gap_summary.items():
    print(f'  {rt}: n={stats["count"]:,}, median gap={stats["median_gap_pct"]}%, >50% gap={stats["pct_gt_50"]}% of records')
print()
if savings_comparison:
    print('=== DOGE Savings vs Actual Ceiling-Obligation Gap ===')
    print(f'  n={savings_comparison["n"]:,}')
    print(f'  Median DOGE savings:   ${savings_comparison["median_doge_savings"]:,.0f}')
    print(f'  Median actual gap:     ${savings_comparison["median_actual_gap"]:,.0f}')
    print(f'  Total DOGE savings:    ${savings_comparison["total_doge_savings"]:,.0f}')
    print(f'  Total actual gap:      ${savings_comparison["total_actual_gap"]:,.0f}')
    print()
print('=== Aggregate Totals ===')
for label, agg in [('Contracts', agg_contracts), ('Grants', agg_grants), ('All awards', agg_all)]:
    print(f'  {label}:')
    print(f'    Ceiling:   ${(agg["total_ceiling"] or 0):,.0f}')
    print(f'    Obligated: ${(agg["total_obligated"] or 0):,.0f}')
    print(f'    Outlays:   ${(agg["total_outlays"] or 0):,.0f}')

## 5. Publish

Builds summary JSON (aggregate statistics) and findings MD. The full per-record dataset is not published — it is too large for GitHub and available locally from the JSONL checkpoints.

In [ ]:
# ── Build summary JSON ─────────────────────────────────────────────────────────
output = {
    'source_notebooks': ['02_fetch'],
    'contracts': {
        'total_assembled':         n_contracts,
        'all_three_numbers':       c_all_three,
        'all_three_numbers_pct':   round(_safe_pct(c_all_three, n_contracts), 1),
        'near_zero_obligation':    c_near_zero_obl,
        'near_zero_obligation_pct': round(_safe_pct(c_near_zero_obl, n_contracts), 1),
        'ceiling_gap_by_type':     gap_summary,
        'aggregate': agg_contracts,
    },
    'grants': {
        'total_assembled':       n_grants,
        'all_three_numbers':     g_all_three,
        'all_three_numbers_pct': round(_safe_pct(g_all_three, n_grants), 1),
        'aggregate': agg_grants,
    },
    'all_awards': {
        'aggregate': agg_all,
    },
    'doge_savings_vs_gap': savings_comparison if savings_comparison else 'savings column not present in matched data',
    'methodology_notes': [
        'ceiling = DOGE value field (contractual maximum including unexercised options)',
        'obligated = USASpending Award Amount (contracts) or child_award_total_obligation (IDVs)',
        'outlays = USASpending Total Outlays (contracts) or child_award_total_outlay (IDVs)',
        'DOGE savings = ceiling minus current obligations; unexercised headroom, not recovered funds',
        'IDV obligation fields at the top-level award record are $0 by design; child task order totals used',
        'Grant linkage covers only the 78% of DOGE grants with a direct USASpending link',
    ],
}

output_json_path.write_text(json.dumps(output, indent=2))
print(json.dumps(output, indent=2))

In [ ]:
# ── Build findings MD ─────────────────────────────────────────────────────────
gap_lines = '\n'.join(
    f'  - {rt}: median gap {s["median_gap_pct"]}%, {s["pct_gt_50"]}% of records show >50% unexercised ceiling (n={s["count"]:,})'
    for rt, s in gap_summary.items()
) or '  - Ceiling gap data not available'

savings_lines = ''
if savings_comparison:
    savings_lines = (
        f'- **DOGE savings vs actual gap (n={savings_comparison["n"]:,}):** '
        f'Total DOGE-reported savings: ${savings_comparison["total_doge_savings"]:,.0f}. '
        f'Total ceiling-minus-obligation gap: ${savings_comparison["total_actual_gap"]:,.0f}. '
        f'DOGE `savings` = ceiling minus current obligations (unexercised headroom); '
        f'termination costs for in-progress work are not netted.'
    )
else:
    savings_lines = '- **DOGE savings comparison:** `savings` column not present in matched checkpoint data.'

findings_md = f"""# Findings — 03: Three-Number Record Assembly

*Input: {n_contracts:,} matched contracts, {n_grants:,} matched grants (from notebook 02 checkpoints).*  
*Methodology: `notebooks/03_assembly.ipynb`*

## Confirmed

- **Three-number coverage — contracts:** {c_all_three:,}/{n_contracts:,} ({_safe_pct(c_all_three, n_contracts):.1f}%) have ceiling, obligated, and outlays present.
- **Three-number coverage — grants:** {g_all_three:,}/{n_grants:,} ({_safe_pct(g_all_three, n_grants):.1f}%) have all three numbers present.
- **Near-zero obligation at cancellation:** {c_near_zero_obl:,} contracts ({_safe_pct(c_near_zero_obl, n_contracts):.1f}%) had obligated amount below 1% of ceiling.
- **Ceiling gap distribution by record type:**
{gap_lines}
{savings_lines}
- **Aggregate totals — contracts:** ceiling ${(agg_contracts['total_ceiling'] or 0):,.0f} / obligated ${(agg_contracts['total_obligated'] or 0):,.0f} / outlays ${(agg_contracts['total_outlays'] or 0):,.0f}
- **Aggregate totals — grants:** ceiling ${(agg_grants['total_ceiling'] or 0):,.0f} / obligated ${(agg_grants['total_obligated'] or 0):,.0f} / outlays ${(agg_grants['total_outlays'] or 0):,.0f}
- **Aggregate totals — all awards:** ceiling ${agg_all['total_ceiling']:,.0f} / obligated ${agg_all['total_obligated']:,.0f} / outlays ${agg_all['total_outlays']:,.0f}

**Methodology constraints carried forward:**
- `ceiling`, `obligated`, and `outlays` are not interchangeable. Ceiling is the contractual maximum including unexercised options. Obligation is cumulative legally committed spend. Outlays are actual disbursements.
- IDV obligation data comes from child task order aggregates via `/api/v2/idvs/amounts/`. Top-level IDV obligation fields in USASpending are $0 by design and are not used.
- Grant coverage is limited to the approximately 78% of DOGE grant records with a direct USASpending link. The remaining 22% have no automated linkage path confirmed in this pipeline.
- USASpending data quality is limited for certain agencies (Agriculture, DHS, GSA, VA, Justice, Interior, OPM). Aggregate totals reflect only what USASpending returned — not total program spend.

## Open

- Audit coverage cross-reference — notebook 04 or later. Note: GAO/IG findings are not joinable to individual award IDs; any cross-reference is probabilistic at agency + program area + timeframe granularity only.
- Linkage path for grants with no `link` host (approx. 22% of DOGE grant records) — unresolved.
- Agency-level breakdown of the three-number record — not included in this summary output; available from local JSONL data.
"""

findings_md_path.write_text(findings_md)
print(findings_md)

Verify output above, then push to GitHub.

In [ ]:
from scripts.colab_utils import publish_artifacts

# Raw JSONL files are gitignored — too large for GitHub, persist in Colab across crashes.
# Only publish derived outputs: summary JSON and findings.
publish_artifacts(
    paths=[
        'data/outputs/03_assembly.json',
        'data/findings/03_assembly.md',
    ],
    message='Three-number record assembly',
    repo_dir=REPO,
)